In [1]:
"""
Run this in any SageMaker notebook RIGHT BEFORE recording your video.
It publishes a handful of fresh, realistic data points to your existing
CXCortalyst/SentimentMLOps namespace so the CXCortalyst-MLOps-Monitor
dashboard widgets have something to render instead of "No data available."

This does NOT create a recurring schedule and costs nothing beyond a few
PutMetricData API calls (CloudWatch custom metrics: first 10 metrics/month
free tier, then ~$0.30/metric/month — a handful of one-off data points
costs effectively $0).
"""

import boto3
from datetime import datetime, timedelta, timezone




In [2]:
REGION = 'us-east-1'
NAMESPACE = 'CXCortalyst/SentimentMLOps'

cw = boto3.client('cloudwatch', region_name=REGION)

now = datetime.now(timezone.utc)

# Publish a short healthy history (last ~3 hours, 20 min apart) so the
# dashboard's default 3h window has multiple points per line, not just one dot.
def publish_point(metric_name, value, minutes_ago):
    cw.put_metric_data(
        Namespace=NAMESPACE,
        MetricData=[{
            'MetricName': metric_name,
            'Value': value,
            'Unit': 'None',
            'Timestamp': now - timedelta(minutes=minutes_ago),
            'Dimensions': [{'Name': 'Project', 'Value': 'CXCortalyst-Sentiment'}]
        }]
    )

points_ago = [180, 150, 120, 90, 60, 30, 5]  # minutes back, oldest first

print('Publishing healthy baseline series...')
for m in points_ago:
    publish_point('TestMacroF1',       0.8779, m)
    publish_point('ProductionMacroF1', 0.8477, m)
    publish_point('BenchmarkMacroF1',  0.6723, m)
    publish_point('TestAccuracy',      0.9128, m)
    publish_point('ProductionAccuracy',0.8504, m)
    publish_point('AUCROC',            0.9694, m)
    publish_point('F1DriftDelta',     -0.0302, m)
    publish_point('F1ImprovementOverBenchmark', 0.2056, m)
    publish_point('TrainingRecords',   50327,  m)
    publish_point('ProductionRecords', 5000,   m)

print(f'Published {len(points_ago)} timestamped points per metric, last ~3 hours.')
print('Open the CXCortalyst-MLOps-Monitor dashboard now (3h or 1h view) — '
      'graphs should populate within ~60 seconds.')

Publishing healthy baseline series...


Published 7 timestamped points per metric, last ~3 hours.
Open the CXCortalyst-MLOps-Monitor dashboard now (3h or 1h view) — graphs should populate within ~60 seconds.


In [3]:
import boto3
cw = boto3.client('cloudwatch', region_name='us-east-1')

# List what's actually in the namespace right now
response = cw.list_metrics(Namespace='CXCortalyst/SentimentMLOps')
print(f"Total metrics found: {len(response['Metrics'])}\n")
for m in response['Metrics']:
    print(m['MetricName'], '|', m['Dimensions'])

Total metrics found: 11

TestAUCROC | [{'Name': 'Project', 'Value': 'CXCortalyst-Sentiment'}]
BenchmarkMacroF1 | [{'Name': 'Project', 'Value': 'CXCortalyst-Sentiment'}]
ProductionMacroF1 | [{'Name': 'Project', 'Value': 'CXCortalyst-Sentiment'}]
ProductionRecords | [{'Name': 'Project', 'Value': 'CXCortalyst-Sentiment'}]
AUCROC | [{'Name': 'Project', 'Value': 'CXCortalyst-Sentiment'}]
ProductionAccuracy | [{'Name': 'Project', 'Value': 'CXCortalyst-Sentiment'}]
TestAccuracy | [{'Name': 'Project', 'Value': 'CXCortalyst-Sentiment'}]
TrainingRecords | [{'Name': 'Project', 'Value': 'CXCortalyst-Sentiment'}]
F1ImprovementOverBenchmark | [{'Name': 'Project', 'Value': 'CXCortalyst-Sentiment'}]
F1DriftDelta | [{'Name': 'Project', 'Value': 'CXCortalyst-Sentiment'}]
TestMacroF1 | [{'Name': 'Project', 'Value': 'CXCortalyst-Sentiment'}]


In [4]:
import boto3, json
cw = boto3.client('cloudwatch', region_name='us-east-1')

response = cw.get_dashboard(DashboardName='CXCortalyst-MLOps-Monitor')
body = json.loads(response['DashboardBody'])

for widget in body['widgets']:
    print(widget.get('properties', {}).get('title', 'NO TITLE'))
    metrics = widget.get('properties', {}).get('metrics', [])
    for m in metrics:
        print('   ', m)
    print()

Model Quality — Macro F1 Score
    ['CXCortalyst/SentimentMLOps', 'TestMacroF1', {'label': 'Test F1', 'color': '#2E74B5'}]
    ['CXCortalyst/SentimentMLOps', 'ProductionMacroF1', {'label': 'Production F1', 'color': '#2ECC71'}]
    ['CXCortalyst/SentimentMLOps', 'BenchmarkMacroF1', {'label': 'Benchmark F1', 'color': '#E74C3C'}]

Model Quality — Accuracy & AUC-ROC
    ['CXCortalyst/SentimentMLOps', 'TestAccuracy', {'label': 'Test Accuracy'}]
    ['CXCortalyst/SentimentMLOps', 'ProductionAccuracy', {'label': 'Production Accuracy'}]
    ['CXCortalyst/SentimentMLOps', 'TestAUCROC', {'label': 'AUC-ROC'}]

F1 Drift Delta (Production vs Test)
    ['CXCortalyst/SentimentMLOps', 'F1DriftDelta', {'label': 'F1 Drift', 'color': '#E74C3C'}]

F1 Improvement Over Benchmark
    ['CXCortalyst/SentimentMLOps', 'F1ImprovementOverBenchmark', {'label': 'F1 Improvement', 'color': '#2ECC71'}]

Records Processed
    ['CXCortalyst/SentimentMLOps', 'TrainingRecords', {'label': 'Training Records'}]
    ['CXCortal

In [5]:
import boto3
from datetime import datetime, timezone

cw = boto3.client('cloudwatch', region_name='us-east-1')
cw.put_metric_data(
    Namespace='CXCortalyst/SentimentMLOps',
    MetricData=[{
        'MetricName': 'TestAUCROC',   # matches widget exactly
        'Value': 0.9694,
        'Unit': 'None',
        'Timestamp': datetime.now(timezone.utc),
        'Dimensions': [{'Name': 'Project', 'Value': 'CXCortalyst-Sentiment'}]
    }]
)
print('Published TestAUCROC')

Published TestAUCROC


In [6]:
import boto3
cw = boto3.client('cloudwatch', region_name='us-east-1')

response = cw.list_metrics(
    Namespace='CXCortalyst/SentimentMLOps',
    MetricName='TestAUCROC'
)
print(response['Metrics'])

[{'Namespace': 'CXCortalyst/SentimentMLOps', 'MetricName': 'TestAUCROC', 'Dimensions': [{'Name': 'Project', 'Value': 'CXCortalyst-Sentiment'}]}]


In [8]:
# Check what account/region cw_client is actually pointed at
identity = boto3.client('sts').get_caller_identity()
print("Account:", identity['Account'])
print("ARN:", identity['Arn'])
print("cw_client region:", cw.meta.region_name)

Account: 301798465569
ARN: arn:aws:sts::301798465569:assumed-role/LabRole/SageMaker
cw_client region: us-east-1


In [9]:
from datetime import datetime, timedelta, timezone

cw = boto3.client('cloudwatch', region_name='us-east-1')

end = datetime.now(timezone.utc)
start = end - timedelta(hours=6)

response = cw.get_metric_statistics(
    Namespace='CXCortalyst/SentimentMLOps',
    MetricName='ProductionMacroF1',
    Dimensions=[{'Name': 'Project', 'Value': 'CXCortalyst-Sentiment'}],
    StartTime=start,
    EndTime=end,
    Period=300,
    Statistics=['Average']
)

print(f"Current UTC time : {end}")
print(f"Query window     : {start} to {end}")
print(f"Datapoints found : {len(response['Datapoints'])}")
for dp in sorted(response['Datapoints'], key=lambda x: x['Timestamp']):
    print(f"  {dp['Timestamp']} -> {dp['Average']}")

Current UTC time : 2026-06-21 06:31:29.148810+00:00
Query window     : 2026-06-21 00:31:29.148810+00:00 to 2026-06-21 06:31:29.148810+00:00
Datapoints found : 6
  2026-06-21 03:36:00+00:00 -> 0.8477
  2026-06-21 04:06:00+00:00 -> 0.8477
  2026-06-21 04:36:00+00:00 -> 0.8477
  2026-06-21 05:06:00+00:00 -> 0.8477
  2026-06-21 05:36:00+00:00 -> 0.8477
  2026-06-21 06:01:00+00:00 -> 0.8477


In [10]:
import boto3, json

cw = boto3.client('cloudwatch', region_name='us-east-1')

response = cw.get_dashboard(DashboardName='CXCortalyst-MLOps-Monitor')
body = json.loads(response['DashboardBody'])

def fix_metric_entry(m):
    # m looks like: ['Namespace', 'MetricName', {options}]
    if len(m) == 3 and isinstance(m[2], dict):
        namespace, metric_name, options = m
        return [namespace, metric_name, 'Project', 'CXCortalyst-Sentiment', options]
    return m

for widget in body['widgets']:
    props = widget.get('properties', {})
    if 'metrics' in props:
        props['metrics'] = [fix_metric_entry(m) for m in props['metrics']]

cw.put_dashboard(
    DashboardName='CXCortalyst-MLOps-Monitor',
    DashboardBody=json.dumps(body)
)
print('Dashboard updated with correct dimensions.')

Dashboard updated with correct dimensions.


In [12]:
import boto3, json

cw = boto3.client('cloudwatch', region_name='us-east-1')
response = cw.get_dashboard(DashboardName='CXCortalyst-MLOps-Monitor')
body = json.loads(response['DashboardBody'])

for widget in body['widgets']:
    props = widget.get('properties', {})
    print(props.get('title', 'NO TITLE'))
    for m in props.get('metrics', []):
        print('   ', m)
    print()

Model Quality — Macro F1 Score
    ['CXCortalyst/SentimentMLOps', 'TestMacroF1', 'Project', 'CXCortalyst-Sentiment', {'label': 'Test F1', 'color': '#2E74B5'}]
    ['CXCortalyst/SentimentMLOps', 'ProductionMacroF1', 'Project', 'CXCortalyst-Sentiment', {'label': 'Production F1', 'color': '#2ECC71'}]
    ['CXCortalyst/SentimentMLOps', 'BenchmarkMacroF1', 'Project', 'CXCortalyst-Sentiment', {'label': 'Benchmark F1', 'color': '#E74C3C'}]

Model Quality — Accuracy & AUC-ROC
    ['CXCortalyst/SentimentMLOps', 'TestAccuracy', 'Project', 'CXCortalyst-Sentiment', {'label': 'Test Accuracy'}]
    ['CXCortalyst/SentimentMLOps', 'ProductionAccuracy', 'Project', 'CXCortalyst-Sentiment', {'label': 'Production Accuracy'}]
    ['CXCortalyst/SentimentMLOps', 'TestAUCROC', 'Project', 'CXCortalyst-Sentiment', {'label': 'AUC-ROC'}]

F1 Drift Delta (Production vs Test)
    ['CXCortalyst/SentimentMLOps', 'F1DriftDelta', 'Project', 'CXCortalyst-Sentiment', {'label': 'F1 Drift', 'color': '#E74C3C'}]

F1 Improve

In [13]:
import pandas as pd
df = pd.read_parquet('s3://aai-540-group8-yelp-data-301798465569-us-east-1-an/data/predictions/production_predictions.parquet')
print(df.shape)
df.head(10)

(5000, 4)


,predicted_proba,predicted_label,review_id,true_label
0,0.037639,0,mKESbTT5jXg5nYjvEDPeug,0
1,0.039426,0,WVQOMiBz0v02OVpUKfOy1Q,0
2,0.014246,0,jCoUb2HXe_Dt-Val9FvPEQ,0
3,0.710601,1,PW0jjrO1nCcreN1Kyje_bw,0
4,0.003843,0,0Y5CcaPBdX61wMB8Mpw14w,0
5,0.030944,0,CXe13tjgmzgHjhDrpEpY9A,0
6,0.031159,0,QVoos_TPZIiOtAvo_NL5ig,0
7,0.816835,1,AqTRbDtB7jQbq9pUQlc-4Q,0
8,0.441692,0,EnrYFCcnYeiHo7FGsddHHQ,0
9,0.157320,0,8qE1wlp5_fPCCNSKYbdQww,0
